In [1]:
import os
import sys

# Add local directories to sys.path
sys.path.append(os.path.abspath("../data"))
sys.path.append(os.path.abspath("../network"))

import torch
import numpy as np

# Local imports
import load
from load import ParallelMelspecDataset
from modules import KameBlock

from typing import Union, List, Tuple, Optional

from torch import nn, Tensor

In [2]:
dataset = ParallelMelspecDataset(melspec_dir = "../../Data/processed/VCTK/melspec", 
                                 transcript_dir = "../../Data/processed/VCTK/transcript_standardized",
                                 speaker_properties_path = "../../Data/processed/VCTK/speaker_properties.csv")

dataset.init_iter_idxs()

In [3]:
sample = dataset[6]
src_melspec, tgt_melspec, tgt_gender = sample

print(src_melspec.shape)

(80, 358)


In [7]:
kameblock = KameBlock(in_ch = 80,
                      conv_ch = 120,
                      out_ch = 80,
                      embed_ch = 2,
                      num_classes = 2)

seq_len = 10

splits = np.split(src_melspec[None,:], np.arange(seq_len, src_melspec.shape[1], step=seq_len), axis=2)

print([x.shape for x in splits])

output = []

for inp in splits:
  output.append(kameblock.forward(torch.tensor(inp), 1))

print([np.shape(x) for x in output])

[(1, 80, 10), (1, 80, 10), (1, 80, 10), (1, 80, 10), (1, 80, 10), (1, 80, 10), (1, 80, 10), (1, 80, 10), (1, 80, 10), (1, 80, 10), (1, 80, 10), (1, 80, 10), (1, 80, 10), (1, 80, 10), (1, 80, 10), (1, 80, 10), (1, 80, 10), (1, 80, 10), (1, 80, 10), (1, 80, 10), (1, 80, 10), (1, 80, 10), (1, 80, 10), (1, 80, 10), (1, 80, 10), (1, 80, 10), (1, 80, 10), (1, 80, 10), (1, 80, 10), (1, 80, 10), (1, 80, 10), (1, 80, 10), (1, 80, 10), (1, 80, 10), (1, 80, 10), (1, 80, 8)]
[torch.Size([1, 80, 10]), torch.Size([1, 80, 10]), torch.Size([1, 80, 10]), torch.Size([1, 80, 10]), torch.Size([1, 80, 10]), torch.Size([1, 80, 10]), torch.Size([1, 80, 10]), torch.Size([1, 80, 10]), torch.Size([1, 80, 10]), torch.Size([1, 80, 10]), torch.Size([1, 80, 10]), torch.Size([1, 80, 10]), torch.Size([1, 80, 10]), torch.Size([1, 80, 10]), torch.Size([1, 80, 10]), torch.Size([1, 80, 10]), torch.Size([1, 80, 10]), torch.Size([1, 80, 10]), torch.Size([1, 80, 10]), torch.Size([1, 80, 10]), torch.Size([1, 80, 10]), torch.

In [6]:
class TestModel(KameBlock):
  def __init__(self, *args, **kwargs):
    super(TestModel, self).__init__(*args, **kwargs)
    
    self.loss_fn = nn.MSELoss()
    
  def train(self, src_melspecs: List[Tensor], tgt_melspecs: List[Tensor], tgt_genders: List[int]):
    
    src_melspecs = self.fit_to_tgt_length(src_melspecs, map(len, tgt_melspecs))
    
    
    
    pred_melspecs = self(src_melspecs, tgt_genders)
    loss = self.loss_fn(pred_melspecs, tgt_melspecs)
    loss.backward()
    
    return loss.detach().numpy()
    
    
model = TestModel(in_ch = 80,
                  conv_ch = 120,
                  out_ch = 80,
                  embed_ch = 2,
                  num_classes = 2)

